#### Step 2: Language Validation

**The problem:** our folder names (`sangraha/verified/hin/`, `sangraha/verified/mar/`, etc.) tell us what language a shard is *supposed* to contain, but not what it *actually* contains. This matters most for Hindi and Marathi because both use the Devanagari script, so a simple Unicode script check would say both are "Devanagari" and cannot tell them apart. A row that is actually Marathi could be sitting in the Hindi folder (or the other way around), and we would never know unless we check the text itself.

**What we are solving:** run every row through a real language identification model (fastText `lid.176`), which looks at actual words and subwords, not just script. For each row we get:

```
predicted_language, language_confidence
```

We then compare `predicted_language` against the `language` field the row already has from Step 1. A mismatch, or a very low confidence score, means the row is suspect and can be reviewed or dropped later. This notebook does not drop anything, it only tags every row so we can decide the filtering rule afterward.

In [2]:
import time
from pathlib import Path

import fasttext
import pandas as pd
from tqdm.auto import tqdm

DATASET_ROOT = Path("../../../../dataset").resolve()
MODEL_PATH = DATASET_ROOT / ".models" / "lid.176.bin"
INPUT_ROOT = DATASET_ROOT / "standardized"
OUTPUT_ROOT = DATASET_ROOT / "lang_validated"
CONFIDENCE_THRESHOLD = 0.70

model = fasttext.load_model(str(MODEL_PATH))

#### Predict function

We only feed the model the first 1000 characters of each document. That is plenty of signal for language identification and keeps this fast over millions of rows.

In [3]:
def predict(text):
    first_chunk = text[:1000].replace("\n", " ").strip()
    if not first_chunk:
        return None, 0.0
    labels, probs = model.predict(first_chunk, k=1)
    return labels[0].replace("__label__", ""), float(probs[0])

#### Find every standardized shard

This walks `dataset/standardized/<language>/<source>/*.parquet`, produced by Step 1.

In [4]:
shards = sorted(INPUT_ROOT.glob("*/*/*.parquet"))
print(f"found {len(shards)} shard(s) to validate")

found 149 shard(s) to validate


#### Run validation on every shard

For each shard: predict language + confidence per row, add those two columns, write to `dataset/lang_validated/<language>/<source>/`. This covers the full dataset (about 29.5M rows), expect roughly 25 to 30 minutes.

In [5]:
summary_rows = []
run_start = time.time()

shard_bar = tqdm(shards, desc="shards", unit="shard")
for shard in shard_bar:
    language = shard.parent.parent.name
    source = shard.parent.name
    shard_bar.set_postfix(current=f"{language}/{source}/{shard.name}")

    df = pd.read_parquet(shard)
    preds = [predict(t) for t in tqdm(df["text"], desc="rows", unit="row", leave=False)]
    df["predicted_language"] = [p[0] for p in preds]
    df["language_confidence"] = [p[1] for p in preds]

    mismatch = (df["predicted_language"] != df["language"]).mean()
    low_conf = (df["language_confidence"] < CONFIDENCE_THRESHOLD).mean()

    out_dir = OUTPUT_ROOT / language / source
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / shard.name, index=False)

    summary_rows.append({
        "shard": f"{language}/{source}/{shard.name}",
        "rows": len(df),
        "mismatch_rate": mismatch,
        "low_confidence_rate": low_conf,
    })

print(f"\nTotal time: {(time.time() - run_start) / 60:.1f} min")

shards: 100%|██████████| 149/149 [47:45<00:00, 19.23s/shard, current=mr/wikipedia/train-00000-of-00001.parquet]


Total time: 47.8 min


#### Summary across all shards

In [6]:
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_ROOT / "validation_summary.csv", index=False)

total_rows = summary["rows"].sum()
weighted_mismatch = (summary["mismatch_rate"] * summary["rows"]).sum() / total_rows
weighted_low_conf = (summary["low_confidence_rate"] * summary["rows"]).sum() / total_rows

print(f"total rows validated: {total_rows}")
print(f"overall mismatch rate: {weighted_mismatch:.2%}")
print(f"overall low confidence rate (below {CONFIDENCE_THRESHOLD}): {weighted_low_conf:.2%}")
summary.sort_values("mismatch_rate", ascending=False).head(10)

total rows validated: 29456220
overall mismatch rate: 0.17%
overall low confidence rate (below 0.7): 0.36%


,shard,rows,mismatch_rate,low_confidence_rate
148,mr/wikipedia/train-00000-of-00001.parquet,94133,0.008690,0.012185
112,hi/wikipedia/train-00000-of-00002.parquet,81547,0.003335,0.011147
113,hi/wikipedia/train-00001-of-00002.parquet,81546,0.002649,0.006328
30,hi/sangraha/data-25.parquet,174763,0.002466,0.004143
33,hi/sangraha/data-28.parquet,174763,0.002466,0.003994
23,hi/sangraha/data-19.parquet,174763,0.002460,0.004045
46,hi/sangraha/data-4.parquet,174763,0.002443,0.003851
91,hi/sangraha/data-80.parquet,174762,0.002403,0.004166
79,hi/sangraha/data-7.parquet,174762,0.002392,0.004103
69,hi/sangraha/data-60.parquet,160929,0.002386,0.003952
